In [1]:
!pip install statsmodels

In [2]:
import pandas as pd
import numpy as np
import zipfile
import matplotlib.pyplot as plt
import gc
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

In [3]:
all_data_df = pd.read_csv("Data.csv")
ss = pd.read_csv("SampleSubmission.csv")

In [4]:
len(ss)

6014

In [5]:
all_data_df.head()

,date_time,v_red,current,power_factor,kwh,Source,v_blue,v_yellow,consumer_device_9,consumer_device_x
0,2024-07-22 18:20:00,137.65,0.08,0.72,0.000661,consumer_device_10_data_user_1,NaN,NaN,0,10
1,2024-07-22 18:25:00,122.82,0.08,0.73,0.000598,consumer_device_10_data_user_1,NaN,NaN,0,10
2,2024-07-22 18:30:00,119.70,0.08,0.74,0.000591,consumer_device_10_data_user_1,NaN,NaN,0,10
3,2024-07-22 18:35:00,124.53,0.08,0.75,0.000623,consumer_device_10_data_user_1,NaN,NaN,0,10
4,2024-07-22 18:40:00,134.84,0.08,0.74,0.000665,consumer_device_10_data_user_1,NaN,NaN,0,10


Each consumer device represents a power pole, and the data user associated with the device is identified in the ID column. Some data users are connected to three-phase systems, while others have single-phase connections. This distinction is reflected in the dataset:

*    Single-phase houses have only `v_red`.
*    Three-phase houses include `v_red`, `v_blue`, and `v_yellow`

The dataset includes measured values for voltage, current, power factor, and energy consumption (kWh). These columns are redundant  voltage (V), current(A), power factor(PF) because the are used in the calculation for energy consumption (kWh).

*    Single-Phase: kWh = (V * A * PF) / 1000 * time
*    Three-phase:  kWh = (√3 * V * A * PF) / 1000 * time
    *    V is the difference voltage between the lines
    
Only significant data would be the `Source`,`date_time`, `kwh`, `Phase_Type` 

In [6]:
all_data_df["Phase_Type"] = np.where(
    (all_data_df["v_blue"].notna() & (all_data_df["v_blue"] != 0)) | 
    (all_data_df["v_yellow"].notna() & (all_data_df["v_yellow"] != 0)),
    "Three-phase", # any blue or any yellow then it is three phase
    np.where(
        (all_data_df["v_red"].notna() & (all_data_df["v_red"] != 0)),
        "Single-phase", # any red and no blue or yellow then single phase
        "Undefined" # all zero or null no red, blue or yellow
    )
)


In [7]:
# Get the dominant non-undefined phase type for each source
# Undefined phase type is when the power is off
source_phases = all_data_df.groupby('Source')['Phase_Type'].apply(
    lambda x: x[x != "Undefined"].mode()[0] if (x != "Undefined").any() else "Undefined"
)
# Map dominant phase to original data
all_data_df['Phase_Type'] = all_data_df['Source'].map(source_phases)

In [8]:
# Split 'Source' into 'consumer_device_X' and 'data_user_Y'
all_data_df[['consumer_device', 'data_user']] = all_data_df['Source'].str.extract(r'(consumer_device_\d+)_data_user_(\d+)')

# Display the updated DataFrame (optional)
all_data_df.head()

,date_time,v_red,current,power_factor,kwh,Source,v_blue,v_yellow,consumer_device_9,consumer_device_x,Phase_Type,consumer_device,data_user
0,2024-07-22 18:20:00,137.65,0.08,0.72,0.000661,consumer_device_10_data_user_1,NaN,NaN,0,10,Single-phase,consumer_device_10,1
1,2024-07-22 18:25:00,122.82,0.08,0.73,0.000598,consumer_device_10_data_user_1,NaN,NaN,0,10,Single-phase,consumer_device_10,1
2,2024-07-22 18:30:00,119.70,0.08,0.74,0.000591,consumer_device_10_data_user_1,NaN,NaN,0,10,Single-phase,consumer_device_10,1
3,2024-07-22 18:35:00,124.53,0.08,0.75,0.000623,consumer_device_10_data_user_1,NaN,NaN,0,10,Single-phase,consumer_device_10,1
4,2024-07-22 18:40:00,134.84,0.08,0.74,0.000665,consumer_device_10_data_user_1,NaN,NaN,0,10,Single-phase,consumer_device_10,1


In [9]:
# These are the devices that are not in the test SampleSubmission

devices_to_drop = ["consumer_device_3","consumer_device_5","consumer_device_11", "consumer_device_14",
                   "consumer_device_15", "consumer_device_17", "consumer_device_24",
                   "consumer_device_25","consumer_device_27","consumer_device_33","consumer_device_4","consumer_device_9"]

In [10]:
# Filter the DataFrame to include only rows where 'consumer_device' is in the 'devices_to_drop' list.
filtered_df = all_data_df[all_data_df['consumer_device'].isin(devices_to_drop)]

# Now 'filtered_df' contains only the rows you specified.  You can further process or save this DataFrame.
filtered_df.head()

,date_time,v_red,current,power_factor,kwh,Source,v_blue,v_yellow,consumer_device_9,consumer_device_x,Phase_Type,consumer_device,data_user
327816,2023-10-01 14:35:00,169.26,3.50,0.73,0.036038,consumer_device_11_data_user_1,NaN,NaN,0,11,Single-phase,consumer_device_11,1
327817,2023-10-01 14:40:00,169.20,3.15,0.76,0.033755,consumer_device_11_data_user_1,NaN,NaN,0,11,Single-phase,consumer_device_11,1
327818,2023-10-01 14:45:00,168.38,2.58,0.73,0.026427,consumer_device_11_data_user_1,NaN,NaN,0,11,Single-phase,consumer_device_11,1
327819,2023-10-01 14:50:00,168.87,2.52,0.76,0.026952,consumer_device_11_data_user_1,NaN,NaN,0,11,Single-phase,consumer_device_11,1
327820,2023-10-01 14:55:00,168.30,2.47,0.75,0.025981,consumer_device_11_data_user_1,NaN,NaN,0,11,Single-phase,consumer_device_11,1


In [11]:
len(all_data_df)/12

3262296.5833333335

In [12]:
all_data_df['date_time'].nunique()

137803

In [13]:
# Convert 'Datetime' column to datetime objects if it's not already
all_data_df['date_time'] = pd.to_datetime(all_data_df['date_time'])

agg_hourly = all_data_df.groupby(
    ['Source', pd.Grouper(key='date_time', freq='H')]
).agg({
    'v_red': 'mean',
    'v_blue': 'mean',
    'v_yellow': 'mean',
    'current': 'mean',
    'power_factor': 'mean',
    'kwh': 'sum',
    'Phase_Type': 'first',  # Now safe to use first() since phases are source-consistent
    'consumer_device': 'first',
    'data_user': 'first'
    
}).reset_index()

In [14]:
# Load climate data with proper datetime parsing
climate = pd.read_excel(
    'Climate Data/Kalam Climate Data.xlsx',
    engine='openpyxl',
    parse_dates=['Date Time']
)
# Rename climate columns for consistency
climate_rename = {
    'Temperature (°C)': 'temp_c',
    'Dewpoint Temperature (°C)': 'dewpoint_c',
    'U Wind Component (m/s)': 'wind_u_ms',
    'V Wind Component (m/s)': 'wind_v_ms',
    'Total Precipitation (mm)': 'precip_mm',
    'Snowfall (mm)': 'snowfall_mm', 
    'Snow Cover (%)': 'snow_cover_pct'
}
climate = climate.rename(columns=climate_rename)


# Merge climate data using datetime alignment
agg_climate_hourly = agg_hourly.merge(
    climate,
    left_on='date_time',
    right_on='Date Time',
    how='left'
).drop(columns=['Date Time'])  # Remove redundant datetime column



In [15]:
climate = climate.rename(columns=climate_rename)

In [16]:
len(agg_hourly)

3262652

In [17]:
len(agg_climate_hourly)

3262652

In [18]:
agg_climate_hourly.describe()

,date_time,v_red,v_blue,v_yellow,current,power_factor,kwh,temp_c,dewpoint_c,wind_u_ms,wind_v_ms,precip_mm,snowfall_mm,snow_cover_pct
count,3262652,1.151731e+06,1.020811e+06,1.090110e+06,3.262652e+06,3.262652e+06,3.262652e+06,3.262641e+06,3.262641e+06,3.262641e+06,3.262641e+06,3.262641e+06,3.262641e+06,3.262641e+06
mean,2024-03-26 01:04:39.019277056,2.176225e+01,2.017574e+01,1.966414e+01,2.015653e-01,7.011571e-02,2.524083e-02,1.408760e-01,-5.871117e+00,-6.737034e-03,-4.189902e-01,2.200379e-03,1.302549e-03,6.603021e+01
min,2023-06-03 12:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-2.228605e+01,-3.366559e+01,-1.084534e+00,-1.461945e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2023-12-28 00:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-8.901022e+00,-1.562411e+01,-8.741760e-02,-9.102478e-01,9.120000e-07,0.000000e+00,3.906250e-03
50%,2024-03-28 09:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-8.450226e-01,-5.873343e+00,1.081848e-02,-5.739136e-01,1.440000e-04,0.000000e+00,9.997266e+01
75%,2024-06-27 18:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.001266e+01,5.452188e+00,8.479309e-02,-8.517456e-02,1.948032e-03,2.930000e-05,9.997266e+01
max,2024-09-23 23:00:00,2.390433e+02,2.390950e+02,2.390950e+02,4.095333e+01,9.900000e-01,8.450901e+00,2.235194e+01,1.704708e+01,1.187332e+00,1.483383e+00,6.257873e-02,6.250127e-02,9.997266e+01
std,NaN,5.380472e+01,5.226781e+01,5.159932e+01,1.493677e+00,1.932134e-01,1.975101e-01,1.063511e+01,1.198801e+01,2.110909e-01,6.431093e-01,5.658139e-03,5.469234e-03,4.586384e+01


Remove the 11 rows that are missing climate data

In [19]:
columns_to_check = [
    'kwh',
    'Phase_Type',
    'temp_c', 
    'dewpoint_c', 
    'wind_u_ms',
    'wind_v_ms', 
    'precip_mm', 
    'snowfall_mm', 
    'snow_cover_pct'
]
missing_rows = agg_climate_hourly[agg_climate_hourly[columns_to_check].isna().any(axis=1)]

agg_climate_hourly.dropna(subset=columns_to_check, inplace=True) # remove the missing rows

missing_rows

,Source,date_time,v_red,v_blue,v_yellow,current,power_factor,kwh,Phase_Type,consumer_device,data_user,temp_c,dewpoint_c,wind_u_ms,wind_v_ms,precip_mm,snowfall_mm,snow_cover_pct
2736942,consumer_device_3_data_user_1,2023-06-03 12:00:00,26.675714,NaN,NaN,0.011429,0.107143,0.000934,Single-phase,consumer_device_3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2746449,consumer_device_3_data_user_10,2023-06-03 12:00:00,26.675714,NaN,NaN,0.011429,0.105714,0.000921,Single-phase,consumer_device_3,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2755956,consumer_device_3_data_user_11,2023-06-03 12:00:00,NaN,NaN,26.704286,0.012857,0.102857,0.001009,Three-phase,consumer_device_3,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2765463,consumer_device_3_data_user_2,2023-06-03 12:00:00,NaN,NaN,26.704286,0.015714,0.037143,0.000446,Three-phase,consumer_device_3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2774970,consumer_device_3_data_user_3,2023-06-03 12:00:00,26.675714,NaN,NaN,0.014286,0.108571,0.001183,Single-phase,consumer_device_3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2784477,consumer_device_3_data_user_4,2023-06-03 12:00:00,NaN,26.704286,NaN,0.012857,0.108571,0.001066,Three-phase,consumer_device_3,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2793984,consumer_device_3_data_user_5,2023-06-03 12:00:00,NaN,NaN,26.704286,0.012857,0.111429,0.001094,Three-phase,consumer_device_3,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2803491,consumer_device_3_data_user_6,2023-06-03 12:00:00,NaN,26.704286,NaN,0.011429,0.114286,0.000997,Three-phase,consumer_device_3,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2812998,consumer_device_3_data_user_7,2023-06-03 12:00:00,NaN,NaN,26.704286,0.010000,0.102857,0.000785,Three-phase,consumer_device_3,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2822505,consumer_device_3_data_user_8,2023-06-03 12:00:00,26.675714,NaN,NaN,0.014286,0.108571,0.001183,Single-phase,consumer_device_3,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
len(agg_climate_hourly)

3262641

In [ ]:
39147559/12

manually calculate kwh

## Note
Consider shifting the climate data all back by an hour so that all of the climate data can be used and avoid droping columns

In [25]:
min_date = climate['Date Time'].min()
min_date

Timestamp('2023-06-03 13:00:00')

In [26]:
max_date = agg_climate_hourly['date_time'].max()
date_rng = pd.date_range(start=max_date + pd.Timedelta(hours=1),  # Start from next hour
                         end=max_date + pd.Timedelta(days=31), 
                         freq='H')
future_predictions = pd.DataFrame()

for source in agg_climate_hourly['Source'].unique():
    # Create base structure
    source_df = pd.DataFrame({
        'date_time': date_rng,
        'Source': source,
        'Phase_Type': source_phases[source],  # Use established phase type
        'kwh': 0  # Placeholder for predictions
    })
    source_df[['consumer_device', 'data_user']] = source_df['Source'].str.extract(r'(consumer_device_\d+)_data_user_(\d+)')

    
    # Merge climate forecasts
    source_df = source_df.merge(
        climate,
        left_on='date_time',
        right_on='Date Time',
        how='inner'  # Ensures climate data exists for all future dates
    ).drop(columns=['Date Time'])
    
    future_predictions = pd.concat([future_predictions, source_df])

# Final Formatting
next_month_predictors_df = future_predictions[[
    'Source', 'date_time', 'Phase_Type', 'consumer_device', 'data_user',
    'temp_c', 'dewpoint_c', 'wind_u_ms', 'wind_v_ms',
    'precip_mm', 'snowfall_mm', 'snow_cover_pct', 'kwh'
]].reset_index(drop=True)


In [27]:
next_month_predictors_df

,Source,date_time,Phase_Type,consumer_device,data_user,temp_c,dewpoint_c,wind_u_ms,wind_v_ms,precip_mm,snowfall_mm,snow_cover_pct,kwh
0,consumer_device_10_data_user_1,2024-09-24 00:00:00,Single-phase,consumer_device_10,1,6.350473,-0.990332,0.017868,-1.176903,0.000018,0.000000,0.000000,0
1,consumer_device_10_data_user_1,2024-09-24 01:00:00,Single-phase,consumer_device_10,1,7.480341,-2.697684,-0.003983,-1.139629,0.000000,0.000000,0.000000,0
2,consumer_device_10_data_user_1,2024-09-24 02:00:00,Single-phase,consumer_device_10,1,8.362009,-3.480551,-0.002762,-1.053040,0.000000,0.000000,0.000000,0
3,consumer_device_10_data_user_1,2024-09-24 03:00:00,Single-phase,consumer_device_10,1,10.587747,3.500421,-0.034714,-0.873757,0.000000,0.000000,0.000000,0
4,consumer_device_10_data_user_1,2024-09-24 04:00:00,Single-phase,consumer_device_10,1,16.494089,-0.551749,-0.261475,-0.637933,0.000000,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
435235,consumer_device_9_data_user_9,2024-10-24 19:00:00,Single-phase,consumer_device_9,9,-4.819373,-5.280783,0.041763,-1.040543,0.000393,0.000128,7.849609,0
435236,consumer_device_9_data_user_9,2024-10-24 20:00:00,Single-phase,consumer_device_9,9,-5.387305,-5.942435,0.045059,-1.108795,0.000394,0.000129,7.855469,0
435237,consumer_device_9_data_user_9,2024-10-24 21:00:00,Single-phase,consumer_device_9,9,-6.008307,-6.837256,0.015198,-1.194077,0.000395,0.000129,7.859375,0
435238,consumer_device_9_data_user_9,2024-10-24 22:00:00,Single-phase,consumer_device_9,9,-6.535742,-7.863989,-0.028015,-1.230438,0.000397,0.000129,7.863281,0


In [28]:
# # Save Cleaned data
# agg_climate_hourly.to_csv("all_hourly_data_w_climate.csv", index = False)
# next_month_predictors_df.to_csv("next_month_hourly_predictors.csv", index = False)


## START HERE
Need to rewrite the model to take into account the new predictors to find the kwh for next month

In [ ]:
# for all_data["Source"] aggregate by sum on day

import pandas as pd
# Assuming 'all_data_df' is already defined as in your previous code.
# Convert 'Datetime' column to datetime objects if it's not already
all_data_df['date_time'] = pd.to_datetime(all_data_df['date_time'])

# Extract the date part
all_data_df['Date'] = all_data_df['date_time'].dt.date

# Group by 'Source' and 'Date', then sum the 'Load' for each group
aggregated_data = all_data_df.groupby(['Source', 'Date'])['kwh'].sum().reset_index()

# Display the aggregated data
aggregated_data.head()


In [ ]:
# Filter data for consumer_device_10
consumer_10_data = aggregated_data[aggregated_data['Source'].str.contains('consumer_device_10')]

# Create the plot
plt.figure(figsize=(12, 6))
for data_user in consumer_10_data['Source'].unique():
    user_data = consumer_10_data[consumer_10_data['Source'] == data_user]
    plt.plot(user_data['Date'], user_data['kwh'], label=data_user)

plt.xlabel('Date')
plt.ylabel('kwh')
plt.title('kwh Consumption for consumer_device_10 per data_user')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Find the minimum and maximum date_time values
min_date = aggregated_data['Date'].min()
max_date = aggregated_data['Date'].max()

print(f"Minimum date_time: {min_date}")
print(f"Maximum date_time: {max_date}")


In [ ]:
# Fill missing date values with 0 kwh

# Create a date range
date_rng = pd.date_range(start=min_date, end=max_date, freq='D')

# Create an empty DataFrame to store the complete data
complete_data = pd.DataFrame()

# Iterate through each unique 'Source'
for source in aggregated_data['Source'].unique():
    # Extract data for the current 'Source'
    source_data = aggregated_data[aggregated_data['Source'] == source].copy()

    # Convert the source data Date to match the type of date_rng
    source_data['Date'] = pd.to_datetime(source_data['Date'])

    # Create a complete date range for the current 'Source'
    source_date_rng = pd.DataFrame({'Date': date_rng})
    source_date_rng['Source'] = source

    # Merge with the existing data, filling missing 'kwh' values with 0
    source_data = pd.merge(source_date_rng, source_data, on=['Date', 'Source'], how='left')
    source_data['kwh'] = source_data['kwh'].fillna(0)

    # Append to the complete data
    complete_data = pd.concat([complete_data, source_data], ignore_index=True)

In [ ]:
complete_data.head()

In [ ]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

# Function to process and forecast per unique consumer_device_x and data_user_y
def forecast_arima(all_data, forecast_horizon=30, output_template=None):
    # Convert Date column to datetime format
    all_data['Date'] = pd.to_datetime(all_data['Date'])

    # Extract consumer_device_x and data_user_y
    all_data[['consumer_device', 'data_user']] = all_data['Source'].str.extract(r'consumer_device_(\d+)_data_user_(\d+)')

    # Ensure data is sorted by consumer_device, data_user, and Date
    all_data = all_data.sort_values(by=['consumer_device', 'data_user', 'Date'])

    # Store forecasts
    forecast_results = []

    # Process each unique consumer_device_x and data_user_y combination
    for (consumer_device, data_user), group in all_data.groupby(["consumer_device", "data_user"]):
        # Set Date as index
        group = group.set_index("Date")

        # Ensure data is in the correct format
        group = group.asfreq('D').fillna(method='ffill')  # Fill missing dates with last known value

        # Fit ARIMA model
        try:
            model = ARIMA(group["kwh"], order=(5, 1, 0))  # ARIMA(5,1,0) as a baseline
            fitted_model = model.fit()

            # Forecast for the next forecast_horizon days
            forecast_dates = pd.date_range(start=group.index[-1] + pd.Timedelta(days=1),
                                           periods=forecast_horizon, freq='D')
            forecast_values = fitted_model.forecast(steps=forecast_horizon)

            # Store results in required format
            forecast_df = pd.DataFrame({
                "ID": [f"{date.strftime('%Y-%m-%d')}_consumer_device_{consumer_device}_data_user_{data_user}"
                        for date in forecast_dates],
                "kwh": forecast_values
            })

            forecast_results.append(forecast_df)

        except Exception as e:
            print(f"Error processing {consumer_device}_{data_user}: {e}")

    # Combine all forecasts into a single DataFrame
    forecast_df = pd.concat(forecast_results, ignore_index=True)

    # If an output template is provided, align the output format
    if output_template is not None:
        output_template = output_template.drop(columns=['kwh'], errors='ignore')
        final_output = output_template.merge(forecast_df, on='ID', how='left').fillna(0)
    else:
        final_output = forecast_df

    return final_output


In [ ]:
forecast = forecast_arima(all_data=complete_data, forecast_horizon=30, output_template=ss)

In [ ]:
forecast.head()

In [ ]:
# prompt: does forecast["kwh"] contain nans if so replace with 0

# Check for NaN values in the 'kwh' column and replace them with 0
forecast["kwh"] = forecast["kwh"].fillna(0)


In [ ]:
len(complete_data), len(forecast), len(ss)

In [ ]:
forecast.to_csv("forecast.csv", index = False)

In [ ]:
# prompt: list the difference in the ID between forecast and ss

# Assuming 'forecast' and 'ss' DataFrames are already defined as in your provided code.

# Convert 'ID' columns to sets for efficient comparison
forecast_ids = set(forecast['ID'])
ss_ids = set(ss['ID'])

# Find IDs present in forecast but not in ss
forecast_only_ids = forecast_ids - ss_ids

# Find IDs present in ss but not in forecast
ss_only_ids = ss_ids - forecast_ids

# Print the IDs that are in forecast but not in ss
print("IDs in 'forecast' but not in 'ss':")
print(forecast_only_ids)


# Print the IDs that are in ss but not in forecast
print("\nIDs in 'ss' but not in 'forecast':")
print(ss_only_ids)

# Print the number of IDs that differ
print(f"\nNumber of IDs that differ: {len(forecast_only_ids) + len(ss_only_ids)}")


In [ ]:
# prompt: compute RMSE score between forecast and ss

import pandas as pd
from sklearn.metrics import mean_squared_error
import math

# Assuming 'forecast' and 'ss' are DataFrames with a common 'ID' column and a 'kwh' column
# containing the forecast and actual values respectively.

# Merge the forecast and ss DataFrames on the 'ID' column
merged_df = pd.merge(forecast, ss, on='ID', how='left', suffixes=('_forecast', '_actual'))

# Calculate the RMSE
rmse = math.sqrt(mean_squared_error(merged_df['kwh_actual'], merged_df['kwh_forecast']))

print(f"RMSE: {rmse}")


In [ ]:
ss

In [ ]:
forecast